In [2]:
import cv2

cap = cv2.VideoCapture(1)  # Indeks kamera untuk DroidCam

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam!")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Tidak dapat membaca frame!")
        break

    cv2.imshow("DroidCam", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [6]:
import os
import cv2
import pytesseract
import numpy as np
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, InputLayer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import img_to_array  # Tambahkan import ini
from tensorflow.keras.preprocessing import image
# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk melakukan OCR pada gambar KTP
def ocr_on_ktp(image):
    # Mengubah gambar menjadi grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Preprocessing gambar untuk meningkatkan akurasi OCR
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    # Menggunakan Tesseract untuk mengekstrak teks
    text = pytesseract.image_to_string(thresh, lang='eng')

    return text


# Fungsi untuk memuat gambar dan memprediksi label menggunakan CNN model
def predict_with_model(image, model):
    # Mengubah gambar menjadi format yang sesuai untuk prediksi
    img_resized = cv2.resize(image, (224, 224))
    img_array = img_to_array(img_resized)  # Gunakan fungsi img_to_array dari Keras
    img_array = np.expand_dims(img_array, axis=0) / 255.0  # Normalisasi

    # Melakukan prediksi dengan model
    pred = model.predict(img_array)

    # Menampilkan prediksi
    if pred >= 0.5:
        return "Class 1"  # KTP dengan kelas 1
    else:
        return "Class 0"  # KTP dengan kelas 0

# Fungsi untuk memuat model dari folder /content/models
def load_latest_model(model_save_path):
    # Dapatkan semua file .h5 dari folder model_save_path
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    # Pilih model yang terakhir (berdasarkan urutan nama file atau buat logika untuk memilih model)
    model_file = sorted(model_files)[-1]  # Model yang terakhir disimpan
    model_path = os.path.join(model_save_path, model_file)

    # Memuat model
    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model


# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)  # Menggunakan kamera HP (biasanya indeks kamera eksternal adalah 1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam! Pastikan kamera terhubung dan tidak digunakan oleh aplikasi lain.")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

while True:
    # Membaca frame dari kamera DroidCam
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame! Pastikan kamera DroidCam berfungsi dengan baik.")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        # Menunggu input dari pengguna untuk melakukan scanning OCR
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            predicted_class = predict_with_model(frame, model)

            # Menampilkan hasil OCR dan prediksi
            print(f"Teks yang terdeteksi oleh OCR: {ocr_text}")
            print(f"Prediksi untuk gambar KTP: {predicted_class}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"  # Path lokal untuk menyimpan gambar
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()

Model yang dimuat: model_fold_4.h5
Scanning gambar...
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Teks yang terdeteksi oleh OCR: PROVINS! JAWA TIMUR:
KABUPATEN SIDOARIO

NIK : 3b03265032040015

me ARGHKA SYAKIFIAH WIBOWO
TompavTgl Laie © MALANG, 19-12-2004
ee,


Prediksi untuk gambar KTP: Class 0
Gambar disimpan di scanned_ktp.jpg
Scanning gambar...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
Teks yang terdeteksi oleh OCR: PROVINSI JAWA TIMUR
KABUPATEN SIDOARJO

+ 3b031485032040035

AISHKA SYAKIRAK WOO,
‘MALANG, 10-12-2008
PEREMPUAE Get. Dar =~
TAUAN POOUG OOMHES RO?
ceees


Prediksi untuk gambar KTP: Class 0
Gambar disimpan di scanned_ktp.jpg
Scanning gambar...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Teks yang terdeteksi oleh OCR: PROVINS| JAWA TIMUR
KABUPATEN SIDOARJO

sik + 3b031L4S0)2040015

| AISHOCA SYAKIRAH WIBOWO-
Sera MBL AONG, 0-2-2008


Prediksi untuk gambar KTP: Class 0
Gambar disimpan di scanned_ktp.jpg
Scanning gambar...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Teks yang terdeteksi oleh OC

In [2]:
import os
import cv2
import pytesseract
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array  # Import ini untuk mengonversi gambar
from tensorflow.keras.preprocessing import image

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk melakukan OCR pada gambar KTP
def ocr_on_ktp(image):
    # Mengubah gambar menjadi grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Preprocessing gambar untuk meningkatkan akurasi OCR
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

    # Menggunakan Tesseract untuk mengekstrak teks
    text = pytesseract.image_to_string(thresh, lang='eng')

    return text


# Fungsi untuk memuat gambar dan memprediksi label menggunakan CNN model
def predict_with_model(image, model):
    # Mengubah ukuran gambar menjadi 224x224
    img_resized = cv2.resize(image, (224, 224))
    
    # Mengonversi gambar menjadi array dan normalisasi
    img_array = img_to_array(img_resized) / 255.0

    # Memeriksa dimensi gambar yang diproses
    print("Dimensi gambar yang diproses:", img_array.shape)

    # Menambahkan dimensi batch (karena model Keras mengharapkan batch input)
    img_array = np.expand_dims(img_array, axis=0)

    # Melakukan prediksi dengan model
    pred = model.predict(img_array)

    # Menentukan prediksi berdasarkan nilai probabilitas
    if pred >= 0.5:
        return "Class 1"  # KTP dengan kelas 1
    else:
        return "Class 0"  # KTP dengan kelas 0


# Fungsi untuk memuat model dari folder /content/models
def load_latest_model(model_save_path):
    # Dapatkan semua file .h5 dari folder model_save_path
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    # Pilih model yang terakhir disimpan
    model_file = sorted(model_files)[-1]  # Model yang terakhir disimpan
    model_path = os.path.join(model_save_path, model_file)

    # Memuat model
    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model


# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)  # Menggunakan kamera HP (biasanya indeks kamera eksternal adalah 1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam! Pastikan kamera terhubung dan tidak digunakan oleh aplikasi lain.")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

while True:
    # Membaca frame dari kamera DroidCam
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame! Pastikan kamera DroidCam berfungsi dengan baik.")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        # Menunggu input dari pengguna untuk melakukan scanning OCR
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            predicted_class = predict_with_model(frame, model)

            # Menampilkan hasil OCR dan prediksi
            print(f"Teks yang terdeteksi oleh OCR: {ocr_text}")
            print(f"Prediksi untuk gambar KTP: {predicted_class}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"  # Path lokal untuk menyimpan gambar
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()


Model yang dimuat: model_fold_4.h5
Scanning gambar...
Dimensi gambar yang diproses: (224, 224, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 864ms/step
Teks yang terdeteksi oleh OCR: using droidcam.app

PROVINS! JAWA TIMUR
KABUPATEN SIDOARIO

NIK * 350306022004000a

hoe : RERNANDA WIDVADHANA TRAM
Jamal | pawonoco, a
ocde kahaende: Ga. ban =

 PERUMTASMIBLOR IEC” rd
Deed


Prediksi untuk gambar KTP: Class 0
Gambar disimpan di scanned_ktp.jpg
Scanning gambar...
Dimensi gambar yang diproses: (224, 224, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
Teks yang terdeteksi oleh OCR: we

‘:

PROVINS! JAWA TIMUR
KABUPATEN SIDOARJO.

+ 3503060110040003

using droidcam.app


Prediksi untuk gambar KTP: Class 0
Gambar disimpan di scanned_ktp.jpg
Scanning gambar...
Dimensi gambar yang diproses: (224, 224, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Teks yang terdeteksi oleh OCR: PROVINSI JAWA TIMUR
KABUPATEN SIDOARJO

NIK + 3b0338503e040015

Maer AISHKA SYAGRAH WBOWO
Tempay tg Latir | WALANG, 1012-2006
Joris kaipain — : P

EASYOCR

In [1]:
import os
import cv2
import easyocr  # Mengganti pytesseract dengan easyocr
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array  # Import ini untuk mengonversi gambar

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan EasyOCR
def ocr_on_ktp(image):
    # Membuat objek EasyOCR
    reader = easyocr.Reader(['en', 'id'])  # Menggunakan bahasa Inggris, bisa ditambah dengan bahasa lain

    # Menggunakan EasyOCR untuk mengekstrak teks dari gambar
    result = reader.readtext(image)

    # Menggabungkan hasil OCR menjadi satu string
    ocr_text = " ".join([text[1] for text in result])

    return ocr_text


# Fungsi untuk memuat gambar dan memprediksi label menggunakan CNN model
def predict_with_model(image, model):
    # Mengubah ukuran gambar menjadi 224x224
    img_resized = cv2.resize(image, (224, 224))
    
    # Mengonversi gambar menjadi array dan normalisasi
    img_array = img_to_array(img_resized) / 255.0

    # Memeriksa dimensi gambar yang diproses
    print("Dimensi gambar yang diproses:", img_array.shape)

    # Menambahkan dimensi batch (karena model Keras mengharapkan batch input)
    img_array = np.expand_dims(img_array, axis=0)

    # Melakukan prediksi dengan model
    pred = model.predict(img_array)

    # Menentukan prediksi berdasarkan nilai probabilitas
    if pred >= 0.5:
        return "Class 1"  # KTP dengan kelas 1
    else:
        return "Class 0"  # KTP dengan kelas 0


# Fungsi untuk memuat model dari folder /content/models
def load_latest_model(model_save_path):
    # Dapatkan semua file .h5 dari folder model_save_path
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    # Pilih model yang terakhir disimpan
    model_file = sorted(model_files)[-1]  # Model yang terakhir disimpan
    model_path = os.path.join(model_save_path, model_file)

    # Memuat model
    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model


# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)  # Menggunakan kamera HP (biasanya indeks kamera eksternal adalah 1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam! Pastikan kamera terhubung dan tidak digunakan oleh aplikasi lain.")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

while True:
    # Membaca frame dari kamera DroidCam
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame! Pastikan kamera DroidCam berfungsi dengan baik.")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        # Menunggu input dari pengguna untuk melakukan scanning OCR
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            predicted_class = predict_with_model(frame, model)

            # Menampilkan hasil OCR dan prediksi
            print(f"Teks yang terdeteksi oleh OCR: {ocr_text}")
            print(f"Prediksi untuk gambar KTP: {predicted_class}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"  # Path lokal untuk menyimpan gambar
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()

ModuleNotFoundError: No module named 'easyocr'

paddleOCR ngambil semua

In [8]:
import os
import cv2
from paddleocr import PaddleOCR
print("PaddleOCR berhasil diimpor!")
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array  # Import ini untuk mengonversi gambar

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan PaddleOCR
def ocr_on_ktp(image):
    # Membuat objek PaddleOCR
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Menggunakan 'id' untuk Bahasa Indonesia

    # Menggunakan PaddleOCR untuk mengekstrak teks dari gambar
    result = ocr.ocr(image, cls=True)

    # Debugging: Lihat struktur hasil OCR
    print("Hasil OCR:", result)

    # Ekstraksi teks dari hasil OCR
    if result and result[0]:  # Cek jika hasil tidak kosong
        ocr_text = " ".join([line[1][0] for line in result[0]])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."

    return ocr_text

# Fungsi untuk memuat gambar dan memprediksi label menggunakan CNN model
def predict_with_model(image, model):
    # Mengubah ukuran gambar menjadi 224x224
    img_resized = cv2.resize(image, (224, 224))
    
    # Mengonversi gambar menjadi array dan normalisasi
    img_array = img_to_array(img_resized) / 255.0

    # Memeriksa dimensi gambar yang diproses
    print("Dimensi gambar yang diproses:", img_array.shape)

    # Menambahkan dimensi batch (karena model Keras mengharapkan batch input)
    img_array = np.expand_dims(img_array, axis=0)

    # Melakukan prediksi dengan model
    pred = model.predict(img_array)

    # Menentukan prediksi berdasarkan nilai probabilitas
    if pred >= 0.5:
        return "Class 1"  # KTP dengan kelas 1
    else:
        return "Class 0"  # KTP dengan kelas 0


# Fungsi untuk memuat model dari folder /content/models
def load_latest_model(model_save_path):
    # Dapatkan semua file .h5 dari folder model_save_path
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    # Pilih model yang terakhir disimpan
    model_file = sorted(model_files)[-1]  # Model yang terakhir disimpan
    model_path = os.path.join(model_save_path, model_file)

    # Memuat model
    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model


# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)  # Menggunakan kamera HP (biasanya indeks kamera eksternal adalah 1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam! Pastikan kamera terhubung dan tidak digunakan oleh aplikasi lain.")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

while True:
    # Membaca frame dari kamera DroidCam
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame! Pastikan kamera DroidCam berfungsi dengan baik.")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        # Menunggu input dari pengguna untuk melakukan scanning OCR
        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            predicted_class = predict_with_model(frame, model)

            # Menampilkan hasil OCR dan prediksi
            print(f"Teks yang terdeteksi oleh OCR: {ocr_text}")
            print(f"Prediksi untuk gambar KTP: {predicted_class}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"  # Path lokal untuk menyimpan gambar
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()

PaddleOCR berhasil diimpor!


[2025-01-14 13:50:42,566] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model yang dimuat: model_fold_4.h5
Scanning gambar...
[2025/01/14 13:50:53] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', rec_image_

ngambil nik, nama, alamat

In [1]:
import os
import cv2
from paddleocr import PaddleOCR
import re
from tensorflow.keras.models import load_model

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk membersihkan dan menormalisasi teks hasil OCR
def normalize_text(text):
    # Menghapus spasi ekstra dan mengubah ke format yang lebih mudah diproses
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan PaddleOCR
def ocr_on_ktp(image):
    # Membuat objek PaddleOCR
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Menggunakan 'id' untuk Bahasa Indonesia

    # Menggunakan PaddleOCR untuk mengekstrak teks dari gambar
    result = ocr.ocr(image, cls=True)

    # Debugging: Lihat struktur hasil OCR
    print("Hasil OCR (mentah):", result)

    # Ekstraksi teks dari hasil OCR
    if result and result[0]:  # Cek jika hasil tidak kosong
        ocr_text = " ".join([line[1][0] for line in result[0]])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."

    return normalize_text(ocr_text)

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    # Menyaring informasi yang relevan dari hasil OCR menggunakan regex
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    # Ekstraksi NIK (biasanya 16 digit)
    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    # Ekstraksi Nama: CAPSLOCK sampai sebelum deskripsi lain
    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = nama_match.group(1).strip()

    # Ekstraksi Alamat: Lengkap hingga batas deskripsi berikutnya
    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        data['alamat'] = alamat_match.group(1).strip()

    # Ekstraksi Kelurahan/Desa: Berhenti di huruf besar tunggal
    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+ [A-Z])', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = kelurahan_match.group(2).strip()

    # Ekstraksi Kecamatan: Berhenti di huruf besar tunggal
    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+ [A-Z])', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = kecamatan_match.group(1).strip()

    return data



# Fungsi untuk memuat model dari folder /content/models
def load_latest_model(model_save_path):
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    model_file = sorted(model_files)[-1]
    model_path = os.path.join(model_save_path, model_file)

    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model

# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam!")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame!")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan hanya data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()

[2025-01-15 11:51:30,410] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model yang dimuat: model_fold_4.h5
Scanning gambar...
[2025/01/15 11:51:56] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', rec_image_

COBA PADDLE

In [2]:
import os
import cv2
from paddleocr import PaddleOCR
import re
from tensorflow.keras.models import load_model

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk membersihkan dan menormalisasi teks hasil OCR
def normalize_text(text):
    # Menghapus spasi ekstra, karakter tunggal yang tidak relevan, dan noise
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = re.sub(r'\b[A-Z0-9]\b', '', text)  # Menghapus karakter tunggal (seperti "K", "A")
    text = re.sub(r'[^\w\s.,]', '', text)  # Menghapus karakter non-alfanumerik kecuali spasi, titik, dan koma
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan PaddleOCR
def ocr_on_ktp(image):
    # Membuat objek PaddleOCR
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Menggunakan 'id' untuk Bahasa Indonesia

    # Menggunakan PaddleOCR untuk mengekstrak teks dari gambar
    result = ocr.ocr(image, cls=True)

    # Debugging: Lihat struktur hasil OCR
    print("Hasil OCR (mentah):", result)

    # Ekstraksi teks dari hasil OCR
    if result and result[0]:  # Cek jika hasil tidak kosong
        ocr_text = " ".join([line[1][0] for line in result[0]])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."

    return normalize_text(ocr_text)

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    # Menyaring informasi yang relevan dari hasil OCR menggunakan regex
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    # Ekstraksi NIK (biasanya 16 digit)
    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    # Ekstraksi Nama: CAPSLOCK sampai sebelum deskripsi lain
    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = normalize_text(nama_match.group(1))

    # Ekstraksi Alamat: Lengkap hingga batas deskripsi berikutnya
    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        data['alamat'] = normalize_text(alamat_match.group(1))

    # Ekstraksi Kelurahan/Desa: Berhenti di huruf besar tunggal
    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = normalize_text(kelurahan_match.group(2))

    # Ekstraksi Kecamatan: Berhenti di huruf besar tunggal
    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = normalize_text(kecamatan_match.group(1))

    return data

# Fungsi untuk memuat model dari folder /content/models
def load_latest_model(model_save_path):
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    model_file = sorted(model_files)[-1]
    model_path = os.path.join(model_save_path, model_file)

    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model

# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam!")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame!")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan hanya data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()

[2025-01-16 07:45:33,257] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model yang dimuat: model_fold_4.h5
Scanning gambar...
[2025/01/16 07:46:35] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', rec_image_

In [6]:
import os
import cv2
from paddleocr import PaddleOCR
import re
from tensorflow.keras.models import load_model

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk membersihkan dan menormalisasi teks hasil OCR
def normalize_text(text):
    # Menghapus spasi ekstra, karakter tunggal yang tidak relevan, dan noise
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = re.sub(r'\b[A-Z0-9]\b', '', text)  # Menghapus karakter tunggal (seperti "K", "A")
    text = re.sub(r'[^\w\s.,]', '', text)  # Menghapus karakter non-alfanumerik kecuali spasi, titik, dan koma
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan PaddleOCR
def ocr_on_ktp(image):
    # Membuat objek PaddleOCR
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Menggunakan 'id' untuk Bahasa Indonesia

    # Menggunakan PaddleOCR untuk mengekstrak teks dari gambar
    result = ocr.ocr(image, cls=True)

    # Debugging: Lihat struktur hasil OCR
    print("Hasil OCR (mentah):", result)

    # Ekstraksi teks dari hasil OCR
    if result and result[0]:  # Cek jika hasil tidak kosong
        ocr_text = " ".join([line[1][0] for line in result[0]])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."

    return normalize_text(ocr_text)

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    # Menyaring informasi yang relevan dari hasil OCR menggunakan regex
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    # Ekstraksi NIK (biasanya 16 digit)
    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    # Ekstraksi Nama: CAPSLOCK sampai sebelum deskripsi lain
    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = normalize_text(nama_match.group(1))

    # Ekstraksi Alamat: Hapus bagian RT/RW dari alamat
    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        alamat = alamat_match.group(1)
        # Menghapus semua pola RT/RW
        alamat_cleaned = re.sub(r'\b(RT|RW|RTRW|RT/RW|R\.T|R\.W)\b\s*\d+(\s*\d+)?', '', alamat, flags=re.IGNORECASE)
        alamat_cleaned = re.sub(r'\bRTAW\b\s*\d+', '', alamat_cleaned, flags=re.IGNORECASE)  # Pola tambahan untuk RTAW
        data['alamat'] = normalize_text(alamat_cleaned)

    # Ekstraksi Kelurahan/Desa
    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = normalize_text(kelurahan_match.group(2))

    # Ekstraksi Kecamatan
    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = normalize_text(kecamatan_match.group(1))

    return data


# Fungsi untuk memuat model dari folder /content/models
def load_latest_model(model_save_path):
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    model_file = sorted(model_files)[-1]
    model_path = os.path.join(model_save_path, model_file)

    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model

# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam!")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame!")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan hanya data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()

[2025-01-16 07:57:01,906] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model yang dimuat: model_fold_4.h5
Scanning gambar...
[2025/01/16 07:57:27] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', rec_image_

FORMAT PKL

In [11]:
class KTPProcessor:
    def __init__(self, model_save_path):
        self.model_save_path = model_save_path
        self.model = None  # Model akan dimuat nanti

    def normalize_text(self, text):
        # Fungsi normalisasi teks
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\b[A-Z0-9]\b', '', text)
        text = re.sub(r'[^\w\s.,]', '', text)
        text = text.strip()
        return text

    def ocr_on_ktp(self, image):
        # Inisialisasi PaddleOCR
        ocr = PaddleOCR(use_angle_cls=True, lang='id')
        result = ocr.ocr(image, cls=True)

        if result and result[0]:
            ocr_text = " ".join([line[1][0] for line in result[0]])
        else:
            ocr_text = "Tidak ada teks yang terdeteksi."

        return self.normalize_text(ocr_text)

    def extract_data_from_ocr(self, ocr_text):
        data = {
            'nik': None,
            'nama': None,
            'alamat': None,
            'kelurahan': None,
            'kecamatan': None
        }

        # Ekstraksi data dari teks
        nik_match = re.search(r'\d{16}', ocr_text)
        if nik_match:
            data['nik'] = nik_match.group()

        nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
        if nama_match:
            data['nama'] = self.normalize_text(nama_match.group(1))

        alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
        if alamat_match:
            alamat = alamat_match.group(1)
            alamat_cleaned = re.sub(r'\b(RT|RW|RTRW|RT/RW|R\.T|R\.W)\b\s*\d+(\s*\d+)?', '', alamat, flags=re.IGNORECASE)
            alamat_cleaned = re.sub(r'\bRTAW\b\s*\d+', '', alamat_cleaned, flags=re.IGNORECASE)
            data['alamat'] = self.normalize_text(alamat_cleaned)

        kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
        if kelurahan_match:
            data['kelurahan'] = self.normalize_text(kelurahan_match.group(2))

        kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
        if kecamatan_match:
            data['kecamatan'] = self.normalize_text(kecamatan_match.group(1))

        return data

    def load_latest_model(self):
        # Memuat model terbaru
        model_files = [f for f in os.listdir(self.model_save_path) if f.endswith('.h5')]

        if not model_files:
            raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

        model_file = sorted(model_files)[-1]
        model_path = os.path.join(self.model_save_path, model_file)
        self.model = load_model(model_path)
        print(f"Model yang dimuat: {model_file}")

    def process_ktp(self, image_path):
        # Proses gambar KTP: baca gambar, OCR, dan ekstrak data
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Gambar tidak ditemukan di path: {image_path}")

        # Baca gambar
        image = cv2.imread(image_path)

        if image is None:
            raise ValueError("Gambar tidak valid atau gagal dibaca.")

        # Lakukan OCR
        ocr_text = self.ocr_on_ktp(image)

        # Ekstrak data dari teks OCR
        extracted_data = self.extract_data_from_ocr(ocr_text)

        return extracted_data

SAVE PKL

In [22]:
import os
import cv2
from paddleocr import PaddleOCR
import re
import joblib
from tensorflow.keras.models import load_model

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk membersihkan dan menormalisasi teks hasil OCR
def normalize_text(text):
    # Menghapus spasi ekstra, karakter tunggal yang tidak relevan, dan noise
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = re.sub(r'\b[A-Z0-9]\b', '', text)  # Menghapus karakter tunggal (seperti "K", "A")
    text = re.sub(r'[^\w\s.,]', '', text)  # Menghapus karakter non-alfanumerik kecuali spasi, titik, dan koma
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan PaddleOCR
def ocr_on_ktp(image):
    # Membuat objek PaddleOCR
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Menggunakan 'id' untuk Bahasa Indonesia

    # Menggunakan PaddleOCR untuk mengekstrak teks dari gambar
    result = ocr.ocr(image, cls=True)

    # Debugging: Lihat struktur hasil OCR
    print("Hasil OCR (mentah):", result)

    # Ekstraksi teks dari hasil OCR
    if result and result[0]:  # Cek jika hasil tidak kosong
        ocr_text = " ".join([line[1][0] for line in result[0]])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."

    return normalize_text(ocr_text)

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    # Menyaring informasi yang relevan dari hasil OCR menggunakan regex
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    # Ekstraksi NIK (biasanya 16 digit)
    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    # Ekstraksi Nama: CAPSLOCK sampai sebelum deskripsi lain
    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = normalize_text(nama_match.group(1))

    # Ekstraksi Alamat: Hapus bagian RT/RW dari alamat
    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        alamat = alamat_match.group(1)
        # Menghapus semua pola RT/RW
        alamat_cleaned = re.sub(r'\b(RT|RW|RTRW|RT/RW|R\.T|R\.W)\b\s*\d+(\s*\d+)?', '', alamat, flags=re.IGNORECASE)
        alamat_cleaned = re.sub(r'\bRTAW\b\s*\d+', '', alamat_cleaned, flags=re.IGNORECASE)  # Pola tambahan untuk RTAW
        data['alamat'] = normalize_text(alamat_cleaned)

    # Ekstraksi Kelurahan/Desa
    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = normalize_text(kelurahan_match.group(2))

    # Ekstraksi Kecamatan
    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = normalize_text(kecamatan_match.group(1))

    return data

# Fungsi untuk memuat model dari folder
def load_latest_model(model_save_path):
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    model_file = sorted(model_files)[-1]
    model_path = os.path.join(model_save_path, model_file)

    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model

# Fungsi untuk menyimpan model sebagai .pkl
def save_model_as_pkl(model, filename='model.pkl'):
    joblib.dump(model, filename)
    print(f"Model disimpan sebagai {filename}")

# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam!")
    exit()

# Memuat model CNN yang sudah dilatih dari folder /content/models
model = load_latest_model(model_save_path)

# Menyimpan model CNN sebagai .pkl agar bisa digunakan lebih mudah
save_model_as_pkl(model)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame!")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan hanya data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()

[2025-01-16 09:00:15,122] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model yang dimuat: model_fold_4.h5
Model disimpan sebagai model.pkl
Scanning gambar...
[2025/01/16 09:00:36] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\lati

In [36]:
import joblib
import numpy as np
import cv2
from tensorflow.keras.preprocessing import image

# Fungsi untuk memuat model dari file .pkl
def load_model_from_pkl(filename='model.pkl'):
    # Memuat model yang disimpan dalam file .pkl
    model = joblib.load(filename)
    print(f"Model yang dimuat: {filename}")

    # Menangkap gambar dari kamera
    cap = cv2.VideoCapture(1)  # Pastikan ID kamera sesuai dengan perangkat Anda

    if not cap.isOpened():
        print("Tidak dapat mengakses kamera!")
        return
    
    ret, img = cap.read()  # Membaca frame dari kamera
    cap.release()  # Menutup kamera setelah membaca gambar

    if not ret:
        print("Gambar tidak dapat dibaca!")
        return
    
    # Mengubah ukuran gambar menjadi sesuai input model (misalnya 224x224)
    img_resized = cv2.resize(img, (224, 224))  # Mengubah ukuran ke 224x224
    
    # Mengubah gambar ke format RGB dan normalisasi
    img_resized = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)  # Konversi ke RGB
    img_array = np.array(img_resized) / 255.0  # Normalisasi gambar ke rentang [0, 1]
    
    # Menambahkan dimensi batch: shape menjadi (1, 224, 224, 3)
    img_array = np.expand_dims(img_array, axis=0)

    # Menggunakan model untuk melakukan prediksi
    predictions = model.predict(img_array)
    print(f"Prediksi: {predictions}")

    return model

# Memuat model dan melakukan prediksi
load_model_from_pkl('model.pkl')


Model yang dimuat: model.pkl
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 704ms/step
Prediksi: [[0.08405206]]


<Sequential name=sequential_7, built=True>

In [1]:
import joblib
import numpy as np
import cv2
from tensorflow.keras.preprocessing import image

# Fungsi untuk memuat model dari file .pkl
def load_model_from_pkl(filename='model.pkl'):
    # Memuat model yang disimpan dalam file .pkl
    model = joblib.load(filename)
    print(f"Model yang dimuat: {filename}")

    # Menangkap gambar dari kamera
    cap = cv2.VideoCapture(1)  # Pastikan ID kamera sesuai dengan perangkat Anda

    if not cap.isOpened():
        print("Tidak dapat mengakses kamera!")
        return
    
    ret, img = cap.read()  # Membaca frame dari kamera
    cap.release()  # Menutup kamera setelah membaca gambar

    if not ret:
        print("Gambar tidak dapat dibaca!")
        return
    
    # Mengubah ukuran gambar menjadi sesuai input model (misalnya 224x224)
    img_resized = cv2.resize(img, (224, 224))  # Mengubah ukuran ke 224x224
    
    # Mengubah gambar ke format RGB dan normalisasi
    img_resized = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)  # Konversi ke RGB
    img_array = np.array(img_resized) / 255.0  # Normalisasi gambar ke rentang [0, 1]
    
    # Menambahkan dimensi batch: shape menjadi (1, 224, 224, 3)
    img_array = np.expand_dims(img_array, axis=0)

    # Menggunakan model untuk melakukan prediksi
    predictions = model.predict(img_array)
    print(f"Prediksi: {predictions}")

    return model

# Memuat model dan melakukan prediksi
load_model_from_pkl('model.pkl')

d:\asbow kuliah\SMT6\MAGANG\ocr_ktp\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 18 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model yang dimuat: model.pkl
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step
Prediksi: [[0.08405206]]


<Sequential name=sequential_7, built=True>

LOAD PKL

In [2]:
import cv2
import numpy as np
import joblib
from paddleocr import PaddleOCR
import re
import os

# Fungsi untuk membersihkan dan menormalisasi teks hasil OCR
def normalize_text(text):
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = re.sub(r'\b[A-Z0-9]\b', '', text)  # Menghapus karakter tunggal
    text = re.sub(r'[^\w\s.,]', '', text)  # Menghapus karakter non-alfanumerik
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan PaddleOCR
def ocr_on_ktp(image):
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Menggunakan 'id' untuk Bahasa Indonesia
    result = ocr.ocr(image, cls=True)

    if result and result[0]:  # Cek jika hasil tidak kosong
        ocr_text = " ".join([line[1][0] for line in result[0]])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."

    return normalize_text(ocr_text)

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    # Ekstraksi NIK (biasanya 16 digit)
    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    # Ekstraksi Nama
    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = normalize_text(nama_match.group(1))

    # Ekstraksi Alamat
    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        alamat = alamat_match.group(1)
        alamat_cleaned = re.sub(r'\b(RT|RW|RTRW|RT/RW|R\.T|R\.W)\b\s*\d+(\s*\d+)?', '', alamat, flags=re.IGNORECASE)
        data['alamat'] = normalize_text(alamat_cleaned)

    # Ekstraksi Kelurahan/Desa
    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = normalize_text(kelurahan_match.group(2))

    # Ekstraksi Kecamatan
    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = normalize_text(kecamatan_match.group(1))

    return data

# Fungsi untuk memuat model dari file .pkl
def load_model_from_pkl(model_filename='model.pkl'):
    model = joblib.load(model_filename)
    print(f"Model berhasil dimuat dari {model_filename}")
    return model

# Fungsi untuk memulai kamera dan melakukan pemrosesan gambar
def start_camera_and_process(model):
    cap = cv2.VideoCapture(1)  # Membuka kamera

    if not cap.isOpened():
        print("Tidak dapat mengakses kamera!")
        return

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Gagal membaca frame!")
            break

        try:
            # Menampilkan frame video secara real-time
            cv2.imshow("Kamera KTP", frame)

            key = cv2.waitKey(1) & 0xFF

            if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
                break
            elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar, melakukan OCR dan prediksi
                print("Scanning gambar...")

                # Memproses gambar yang diambil dari kamera
                ocr_text = ocr_on_ktp(frame)
                print("Teks hasil OCR yang dinormalisasi:", ocr_text)

                # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
                extracted_data = extract_data_from_ocr(ocr_text)

                # Menampilkan hanya data yang diekstraksi
                print(f"Data yang diekstraksi dari OCR:")
                print(f"NIK: {extracted_data['nik']}")
                print(f"Nama: {extracted_data['nama']}")
                print(f"Alamat: {extracted_data['alamat']}")
                print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
                print(f"Kecamatan: {extracted_data['kecamatan']}")

                # Resize gambar sesuai input model dan lakukan prediksi
                image_resized = cv2.resize(frame, (32, 32))  # Resize sesuai dengan input model
                image_array = np.array(image_resized) / 255.0  # Normalisasi jika diperlukan
                image_array = np.expand_dims(image_array, axis=0)  # Menambahkan dimensi batch

                # Melakukan prediksi dengan model
                predictions = model.predict(image_array)
                print("Hasil Prediksi:", predictions)

                # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
                image_filename = "scanned_ktp.jpg"
                cv2.imwrite(image_filename, frame)
                print(f"Gambar disimpan di {image_filename}")

        except cv2.error as e:
            print(f"OpenCV Error: {e}")
            break

    # Menutup kamera dan jendela OpenCV
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    # Memuat model dari file .pkl
    model = load_model_from_pkl('model.pkl')

    # Memulai kamera dan pemrosesan gambar
    start_camera_and_process(model)


Model berhasil dimuat dari model.pkl
Scanning gambar...
[2025/01/16 09:35:03] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', rec_imag

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense_14" is incompatible with the layer: expected axis -1 of input shape to have value 186624, but received input with shape (1, 2304)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 32, 32, 3), dtype=float32)
  • training=False
  • mask=None

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten
import os

# Model CNN sederhana untuk contoh
def create_model():
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Fungsi untuk menyimpan model ke .h5
def save_model_to_h5(model, filename='model.h5'):
    model.save(filename)
    print(f"Model disimpan sebagai {filename}")

# Membuat model dan menyimpan ke .h5
model = create_model()
save_model_to_h5(model, 'model_ocr_ktp.h5')


[2025-01-16 10:24:32,174] [ WARNING] saving_api.py:83 - You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


Model disimpan sebagai model_ocr_ktp.h5


In [9]:
# Fungsi untuk memuat model dari file .h5
def load_model_from_h5(model_path):
    # Memuat model dari file .h5
    model = load_model(model_path)
    print(f"Model berhasil dimuat dari {model_path}")
    return model

# Fungsi untuk memulai kamera dan melakukan pemindaian KTP
def scan_ktp_with_model(model):
    cap = cv2.VideoCapture(1)  # Mengakses kamera (0 adalah ID default untuk kamera)

    if not cap.isOpened():
        print("Tidak dapat mengakses kamera!")
        return

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Tidak dapat membaca frame!")
            break

        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR
            print("Scanning gambar...")

            # Proses OCR pada gambar
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")

    # Menutup kamera dan jendela OpenCV
    cap.release()
    cv2.destroyAllWindows()

# Nama file model .h5
model_path = 'model_ocr_ktp.h5'  # Ganti dengan lokasi model .h5 yang sesuai

# Memuat model
model = load_model_from_h5(model_path)

# Mulai pemindaian KTP dengan model yang telah dimuat
scan_ktp_with_model(model)

[2025-01-16 10:28:49,496] [ WARNING] saving_utils.py:256 - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


Model berhasil dimuat dari model_ocr_ktp.h5
Scanning gambar...
[2025/01/16 10:29:01] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\user/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\user/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', r

In [2]:
# Fungsi untuk memuat model dari file .h5
def load_model_from_h5(model_path):
    model = load_model(model_path)
    print(f"Model yang dimuat: {model_path}")
    return model

# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam!")
    exit()

# Memuat model CNN yang sudah dilatih dari file .h5
model_path = "model.h5"  # Pastikan file model .h5 sudah ada
model = load_model_from_h5(model_path)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame!")
        break

    try:
        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR dan prediksi
            print("Scanning gambar...")

            # Memproses gambar yang diambil dari kamera
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan hanya data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")
    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [20]:
import pickle
import cv2
import os

# Load pipeline dari file .pkl
with open('ktp_processor.pkl', 'rb') as file:
    ktp_processor = pickle.load(file)

# Membuka kamera (gunakan indeks kamera yang sesuai)
cap = cv2.VideoCapture(1)  # 0 untuk kamera default, jika menggunakan DroidCam bisa ganti dengan 1

if not cap.isOpened():
    print("Tidak dapat mengakses kamera!")
    exit()

while True:
    ret, frame = cap.read()

    if not ret:
        print("Gagal membaca frame!")
        break

    # Menampilkan gambar secara real-time
    cv2.imshow("Camera Scan KTP", frame)

    # Menunggu input dari pengguna
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):  # Tekan 'q' untuk keluar
        break
    elif key == ord('s'):  # Tekan 's' untuk menyimpan gambar dan memprosesnya
        # Menyimpan gambar yang diambil
        image_path = 'scanned_ktp.jpg'
        cv2.imwrite(image_path, frame)
        print(f"Gambar KTP disimpan di {image_path}")

        # Memeriksa apakah gambar berhasil disimpan
        if os.path.exists(image_path):
            print(f"Gambar berhasil disimpan, memeriksa file...")
            image = cv2.imread(image_path)

            if image is None:
                print(f"Error: Gambar tidak dapat dibaca, file mungkin rusak!")
            else:
                print(f"Gambar berhasil dibaca. Ukuran gambar: {image.shape}")
                print(f"Tipe gambar: {type(image)}")
                print(f"Dimensi gambar: {image.shape}")

                # Konversi gambar ke grayscale sebelum diproses (jika diperlukan)
                gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

                try:
                    # Proses gambar dan ekstrak data
                    print("Memproses gambar dengan ktp_processor...")
                    result = ktp_processor.process_ktp(gray_image)  # Menggunakan gambar grayscale

                    # Output hasil
                    print(f"NIK: {result['nik']}")
                    print(f"Nama: {result['nama']}")
                    print(f"Alamat: {result['alamat']}")
                    print(f"Kelurahan: {result['kelurahan']}")
                    print(f"Kecamatan: {result['kecamatan']}")
                except FileNotFoundError as e:
                    print(f"File tidak ditemukan: {e}")
                except ValueError as e:
                    print(f"Error pada gambar: {e}")
                except Exception as e:
                    print(f"Terjadi kesalahan: {e}")
        else:
            print(f"Gambar tidak dapat disimpan! Path: {image_path}")

# Menutup kamera dan jendela OpenCV
cap.release()
cv2.destroyAllWindows()


Gambar KTP disimpan di scanned_ktp.jpg
Gambar berhasil disimpan, memeriksa file...
Gambar berhasil dibaca. Ukuran gambar: (480, 640, 3)
Tipe gambar: <class 'numpy.ndarray'>
Dimensi gambar: (480, 640, 3)
Memproses gambar dengan ktp_processor...
File tidak ditemukan: Gambar tidak ditemukan di path: [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [2]:
import os
import cv2
from paddleocr import PaddleOCR
import re
from tensorflow.keras.models import load_model
from ultralytics import YOLO

# Memuat model YOLOv8
model = YOLO("yolov8n.pt")  # YOLOv8n adalah model kecil, cocok untuk pengujian awal
print("YOLOv8 berhasil diimpor!")


# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk membersihkan dan menormalisasi teks hasil OCR
def normalize_text(text):
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan PaddleOCR
def ocr_on_ktp(image):
    ocr = PaddleOCR(use_angle_cls=True, lang='id')  # Menggunakan 'id' untuk Bahasa Indonesia
    result = ocr.ocr(image, cls=True)

    print("Hasil OCR (mentah):", result)

    if result and result[0]:
        ocr_text = " ".join([line[1][0] for line in result[0]])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."

    return normalize_text(ocr_text)

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = nama_match.group(1).strip()

    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        data['alamat'] = alamat_match.group(1).strip()

    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+ [A-Z])', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = kelurahan_match.group(2).strip()

    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+ [A-Z])', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = kecamatan_match.group(1).strip()

    return data

# Fungsi untuk memuat model CNN
def load_latest_model(model_save_path):
    model_files = [f for f in os.listdir(model_save_path) if f.endswith('.h5')]

    if not model_files:
        raise FileNotFoundError("Tidak ada model .h5 ditemukan dalam folder.")

    model_file = sorted(model_files)[-1]
    model_path = os.path.join(model_save_path, model_file)

    model = load_model(model_path)
    print(f"Model yang dimuat: {model_file}")
    return model

# Fungsi untuk memproses gambar menggunakan YOLO
def detect_ktp_regions_with_yolo(frame, yolo_model):
    results = yolo_model(frame)

    # Memproses hasil deteksi
    boxes = results[0].boxes.xyxy.cpu().numpy()  # Koordinat bounding box
    classes = results[0].boxes.cls.cpu().numpy()  # Kelas objek yang terdeteksi
    scores = results[0].boxes.conf.cpu().numpy()  # Confidence scores

    detected_regions = []

    for box, cls, score in zip(boxes, classes, scores):
        if score > 0.5:  # Filter berdasarkan confidence threshold
            x1, y1, x2, y2 = map(int, box)  # Koordinat bounding box
            detected_regions.append((cls, (x1, y1, x2, y2)))

    return detected_regions

# Membuka kamera DroidCam untuk melakukan scanning KTP
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Tidak dapat mengakses kamera DroidCam!")
    exit()

# Memuat model CNN
model = load_latest_model(model_save_path)

# Memuat model YOLO
yolo_model = YOLO("yolov8n.pt")  # Ganti dengan path ke model YOLO yang sesuai

while True:
    ret, frame = cap.read()

    if not ret:
        print("Tidak dapat membaca frame!")
        break

    try:
        cv2.imshow("Kamera KTP (DroidCam)", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break
        elif key == ord('s'):  # Mulai deteksi dengan YOLO
            print("Scanning gambar...")

            # Deteksi area penting pada KTP menggunakan YOLO
            regions = detect_ktp_regions_with_yolo(frame, yolo_model)

            for cls, (x1, y1, x2, y2) in regions:
                cropped_region = frame[y1:y2, x1:x2]  # Memotong area terdeteksi
                ocr_text = ocr_on_ktp(cropped_region)

                extracted_data = extract_data_from_ocr(ocr_text)

                print(f"Data yang diekstraksi dari area {cls}:")
                print(f"NIK: {extracted_data['nik']}")
                print(f"Nama: {extracted_data['nama']}")
                print(f"Alamat: {extracted_data['alamat']}")
                print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
                print(f"Kecamatan: {extracted_data['kecamatan']}")

    except cv2.error as e:
        print(f"OpenCV Error: {e}")
        break

cap.release()
cv2.destroyAllWindows()

OSError: [WinError 127] The specified procedure could not be found. Error loading "d:\asbow kuliah\SMT6\MAGANG\ocr_ktp\.venv\Lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [1]:
!pip install easyocr


   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   -------------- ------------------------- 1.0/2.9 MB 6.3 MB/s eta 0:00:01
   -------------------------------- ------- 2.4/2.9 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 5.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   --------------------------------- ------ 1.3/1.6 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 5.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/203.1 MB ? eta -:--:--
   ---------------------------------------- 1.3/203.1 MB 6.1 MB/s eta 0:00:34
   ---------------------------------------- 2.1/203.1 MB 5.1 MB/s eta 0:00:40
    --------------------------------------- 3.4/203.1 MB 5.4 MB/s eta 0:00:37
    --------------------------------------- 4.7/203.1 MB 5.7 MB/s eta 0:00:35
   - -------------------------------------- 6.3/203.1 MB 5.8 MB/s eta 0:00:34
   - ----------------

INI BISA EASYOCR

In [4]:
import os
import cv2
import re
import easyocr
from tensorflow.keras.models import load_model

# Folder model yang telah disimpan
model_save_path = '/content/models'

# Fungsi untuk membersihkan dan menormalisasi teks hasil OCR
def normalize_text(text):
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = re.sub(r'\b[A-Z0-9]\b', '', text)  # Menghapus karakter tunggal
    text = re.sub(r'[^\w\s.,]', '', text)  # Menghapus karakter non-alfanumerik kecuali spasi, titik, dan koma
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar KTP menggunakan EasyOCR
def ocr_on_ktp(image):
    reader = easyocr.Reader(['id'])  # Menggunakan 'id' untuk Bahasa Indonesia
    result = reader.readtext(image)
    if result:
        ocr_text = " ".join([item[1] for item in result])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."
    return normalize_text(ocr_text)

# Fungsi untuk ekstrak data dari teks OCR
def extract_data_from_ocr(ocr_text):
    data = {
        'nik': None,
        'nama': None,
        'alamat': None,
        'kelurahan': None,
        'kecamatan': None
    }

    nik_match = re.search(r'\d{16}', ocr_text)
    if nik_match:
        data['nik'] = nik_match.group()

    nama_match = re.search(r'Nama\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if nama_match:
        data['nama'] = normalize_text(nama_match.group(1))

    alamat_match = re.search(r'Alamat\s*[:\-]?\s*([A-Z0-9\s.,]+)', ocr_text)
    if alamat_match:
        alamat = alamat_match.group(1)
        alamat_cleaned = re.sub(r'\b(RT|RW|RTRW|RT/RW|R\.T|R\.W)\b\s*\d+(\s*\d+)?', '', alamat, flags=re.IGNORECASE)
        data['alamat'] = normalize_text(alamat_cleaned)

    kelurahan_match = re.search(r'(Kelurahan|Desa)\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kelurahan_match:
        data['kelurahan'] = normalize_text(kelurahan_match.group(2))

    kecamatan_match = re.search(r'Kecamatan\s*[:\-]?\s*([A-Z\s]+)', ocr_text)
    if kecamatan_match:
        data['kecamatan'] = normalize_text(kecamatan_match.group(1))

    return data

# Fungsi untuk memuat model dari file .h5
def load_model_from_h5(model_path):
    # Memuat model dari file .h5
    model = load_model(model_path)
    print(f"Model berhasil dimuat dari {model_path}")
    return model

# Fungsi untuk memulai kamera dan melakukan pemindaian KTP
def scan_ktp_with_model(model):
    cap = cv2.VideoCapture(1)  # Mengakses kamera (1 adalah ID kamera kedua, sesuaikan jika perlu)

    if not cap.isOpened():
        print("Tidak dapat mengakses kamera!")
        return

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Tidak dapat membaca frame!")
            break

        # Menampilkan frame video secara real-time
        cv2.imshow("Kamera KTP", frame)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):  # Menekan 'q' untuk keluar dari loop
            break
        elif key == ord('s'):  # Menekan 's' untuk menyimpan gambar dan memulai OCR
            print("Scanning gambar...")

            # Proses OCR pada gambar menggunakan EasyOCR
            ocr_text = ocr_on_ktp(frame)
            print("Teks hasil OCR yang dinormalisasi:", ocr_text)

            # Ekstrak data NIK, Nama, Alamat, Kelurahan, Kecamatan dari teks OCR
            extracted_data = extract_data_from_ocr(ocr_text)

            # Menampilkan data yang diekstraksi
            print(f"Data yang diekstraksi dari OCR:")
            print(f"NIK: {extracted_data['nik']}")
            print(f"Nama: {extracted_data['nama']}")
            print(f"Alamat: {extracted_data['alamat']}")
            print(f"Kelurahan/Desa: {extracted_data['kelurahan']}")
            print(f"Kecamatan: {extracted_data['kecamatan']}")

            # Menyimpan gambar untuk referensi lebih lanjut (jika diperlukan)
            image_filename = "scanned_ktp.jpg"
            cv2.imwrite(image_filename, frame)
            print(f"Gambar disimpan di {image_filename}")

    # Menutup kamera dan jendela OpenCV
    cap.release()
    cv2.destroyAllWindows()

# Nama file model .h5
model_path = 'model_ocr_ktp.h5'  # Ganti dengan lokasi model .h5 yang sesuai

# Memuat model
model = load_model_from_h5(model_path)

# Mulai pemindaian KTP dengan model yang telah dimuat
scan_ktp_with_model(model)


Model berhasil dimuat dari model_ocr_ktp.h5


Scanning gambar...
Teks hasil OCR yang dinormalisasi: PROVINSI JAWA Timur KABUPATEN SIDOARJO NIK 3503060110040003 Nama FEANANDA WIDYADHANA ISAQIF TempavTgl Lahir PONOAOGO 01102004 Jonis kolamin LAKILAKI Golstaran Alamal PERuMTAS III BLOK   NO 18 atiaw 057009 KelDesa GAABAGAN Kecamatan TULANGAN Agama ISLAM Slalus Porkawinan BELUM KAWIN Pekerjaan PELAJAAMAHASISWA SiDOAAJO Kewargancoaraan WNI 22102021 Bedaku Hinooa Seumur HiDuP Kariu KARTU UK TAN couK N0 7ru
Data yang diekstraksi dari OCR:
NIK: 3503060110040003
Nama: FEANANDA WIDYADHANA ISAQIF
Alamat: None
Kelurahan/Desa: GAABAGAN
Kecamatan: TULANGAN
Gambar disimpan di scanned_ktp.jpg


Scanning gambar...
Teks hasil OCR yang dinormalisasi: PROVINSI JAWA TimuR KABUPATEN SIDOARJO NIK 3503060110040003 Nama FERNANDA WIDYADHANA TSAQIF TempaTgi Lahir PONOAOGO 0110 2004 Jenis kolamin LAKILAKI Golenaran Alamat PERUMTASIIL BLOK   NO18 atiaw 057009 KelDesa GAABAGAN Kecamatan TULANGAN Agama ISLAM Slatus Perkawinan BELUM KAWIN Pekorjaan PELAMAMAHASISWA SiDoArJo Kewarganegaraan WNI  225102021 Bedaku Hingga Seumur Hicup KARTO VJUKKARTU TAN ru
Data yang diekstraksi dari OCR:
NIK: 3503060110040003
Nama: FERNANDA WIDYADHANA TSAQIF
Alamat: PERUMTASIIL BLOK NO18
Kelurahan/Desa: GAABAGAN
Kecamatan: TULANGAN
Gambar disimpan di scanned_ktp.jpg


Scanning gambar...
Teks hasil OCR yang dinormalisasi: PROVINSI JAWA TiMuR KABUPATEN SIDOARJO NIK 3603185012040015 Name AISHKA SYaKIAAH WIBOWO TompauTal Lahir MALANG, 10122004 Jongkdamın Perevpuan Go. Darah Nlama TaMan PINAnG InDaHBSNo2 ataw 030006 KaDosa Lemahputro Kecamatan SIDOAAJO Agame ISUAM Stalus Pokawnan BELUM KaWIN Pcajaan PELAMAMAHASISWA SiDoAAJO aroancoaruan WNI 02 012022 Bedaku Seumua Hidup 222  5am app Quy KAR  Kaftu UK Hnooa 0l  aru
Data yang diekstraksi dari OCR:
NIK: 3603185012040015
Nama: None
Alamat: None
Kelurahan/Desa: None
Kecamatan: SIDOAAJO
Gambar disimpan di scanned_ktp.jpg


Scanning gambar...
Teks hasil OCR yang dinormalisasi: PROVINSI JAWA TimuR KABUPATEN SIDOARJO NIK 3603185012040015 Nama AISHKA SYAKIRAH WIBOWO TempauTgl Lahir MALANG 10122004 Jonis kolamin PEREMPUAN Gol. Darah Alamal TAMAN PINANG INDAH   NO  ataw 038006 KelDesa LEMAHPUTAO Kecamalan SIDOAAJO Agama ISLAM Stalus Porkawinan BELUM KAWIN Pohoraan PELAJAAMAHASISWA SiDoARJO Kowarganogaraan WNI 03 012022 Berlaku Hingga SeumUR HiDUP 1p 78 y KARTC KAe7o Uk cud Jjx t es
Data yang diekstraksi dari OCR:
NIK: 3603185012040015
Nama: AISHKA SYAKIRAH WIBOWO
Alamat: None
Kelurahan/Desa: LEMAHPUTAO
Kecamatan: None
Gambar disimpan di scanned_ktp.jpg


Scanning gambar...
Teks hasil OCR yang dinormalisasi: PROVINSI JAWA TIMUR KABUPATEN SIDOARJO NIK 3603185012040015 Nama AISHKA SYAKIRAH WIBOWO TempalTol Lahir MALANG 10122004 Jenis kolamin PEREMPUAN Gor. Darah Alamat TAMAN PINANG INDAHB  NO  rTAW 038006 KelDasa LEMaHPUTRO Kocamalan SIDOAAJO Agama ISLAM Stalus Perkawinan BELUM KAWIN Pckorjaan PELAJAAMMAHASISWA SIDOAAJO Kowarganegaraan WNI 03012022 Borlaku Hingoa Seumur HiduP 4F Qhs  32 52535242 222 KARTO KAfto NDUd Jouk rU
Data yang diekstraksi dari OCR:
NIK: 3603185012040015
Nama: AISHKA SYAKIRAH WIBOWO
Alamat: TAMAN PINANG INDAHB NO
Kelurahan/Desa: None
Kecamatan: None
Gambar disimpan di scanned_ktp.jpg


Scanning gambar...
Teks hasil OCR yang dinormalisasi: PROVINSI JAWA Timur KABUPATEN SIDOARJO NIK 3603185012040015 Name AISHKA SYAKIRAH WIBOWO TempaVTo Lahir MALANG 10122004 Jons kdlamın PerempuAn Go. Darah Aamal TAMAN PINANG INDAHB  NO  rtiaw 038006 KeVDesa LeMahPUTRo Kecamalan SIDOAAJO Agama ISLAM Stalus Perkawinan BELUM KAWIN Pckenaan PELAJAAMAHASISWA SIDOAAJO Kovargancoaraan WNI 03012022 Bedaku Hinooa SeumuR HiDUP TP  Ql KARTO KARTU JK Duc Buou Ztu
Data yang diekstraksi dari OCR:
NIK: 3603185012040015
Nama: None
Alamat: None
Kelurahan/Desa: 
Kecamatan: None
Gambar disimpan di scanned_ktp.jpg


Scanning gambar...
Teks hasil OCR yang dinormalisasi: PROVINSI JAWA TimuR KABUPATEN SIDOARJO NIK 3603185012040015 Nama AISHKA SYAKIAAH WIBOWO empauTgl Lahir MALANG 10122004 Joniskolamin Perempuan Go. Darah Nlama TAMAn PINAnG INDAHB  NO  atiaw 038006 KovDesa LeMAHPUTRO Kecamalan SIDOAAJO Agama ISLAM Stalus Porkawnan BELUM KAWIN Pckorjaan PELAMAMAHASISWA SiDOAAJO Kowargancoaraan WNI 03012022 Berlaku Hingoa SeuMUR HiDUP Tp 751 am 35832 y KaRTU KARTU uR NDUD Dun OTU
Data yang diekstraksi dari OCR:
NIK: 3603185012040015
Nama: AISHKA SYAKIAAH WIBOWO
Alamat: None
Kelurahan/Desa: 
Kecamatan: None
Gambar disimpan di scanned_ktp.jpg


Scanning gambar...
Teks hasil OCR yang dinormalisasi: PROVINSI JAWA TimuR KABUPATEN SIDOARJO NIK 3603185012040015 Nama AISHKA SYAKIRAH WIBOWO TempauTg Lahir MALANG 10122004 Jenis kolamin PEREMPUAN Gol. Darah Alamat TAMAN PINANG INDAHB  NO  rtiaw 038006 KeVDesa LEMAHPUTRO Kecamatan SIDOAAJO Agama ISLAM Stalus Perkawinan BELUM KAWIN Pekerjaan PELAJAAMAHASISWA SiDOARJO Kewarganegaraan WNI 03012022 Beraku SeuMUR HiduP  Vul KaR KaRTu Hingoa VouK tu
Data yang diekstraksi dari OCR:
NIK: 3603185012040015
Nama: AISHKA SYAKIRAH WIBOWO
Alamat: TAMAN PINANG INDAHB NO
Kelurahan/Desa: LEMAHPUTRO
Kecamatan: SIDOAAJO
Gambar disimpan di scanned_ktp.jpg


API FLASK

In [ ]:
from flask import Flask, request, jsonify
import easyocr
import re

app = Flask(__name__)

# Fungsi untuk menormalisasi teks hasil OCR
def normalize_text(text):
    text = re.sub(r'\s+', ' ', text)  # Menghapus spasi ganda
    text = re.sub(r'\b[A-Z0-9]\b', '', text)  # Menghapus karakter tunggal
    text = re.sub(r'[^\w\s.,]', '', text)  # Menghapus karakter non-alfanumerik kecuali spasi, titik, dan koma
    text = text.strip()  # Menghapus spasi di awal/akhir
    return text

# Fungsi untuk melakukan OCR pada gambar menggunakan EasyOCR
def ocr_on_ktp(image_path):
    reader = easyocr.Reader(['id'])  # Menggunakan 'id' untuk Bahasa Indonesia
    result = reader.readtext(image_path)
    if result:
        ocr_text = " ".join([item[1] for item in result])
    else:
        ocr_text = "Tidak ada teks yang terdeteksi."
    return normalize_text(ocr_text)

@app.route('/ocr', methods=['POST'])
def process_ocr():
    if 'file' not in request.files:
        return jsonify({'error': 'No file part'}), 400
    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': 'No selected file'}), 400
    
    file_path = f"./uploads/{file.filename}"
    file.save(file_path)
    
    ocr_text = ocr_on_ktp(file_path)
    
    return jsonify({'ocr_text': ocr_text})

if __name__ == '__main__':
    app.run(debug=True)


ONNX

1. Akses dan Ekspor Model CRAFT (Detector):

In [2]:
import easyocr
import torch

# Inisialisasi EasyOCR Reader
reader = easyocr.Reader(['id'])

# Akses model CRAFT dari EasyOCR
craft_model = reader.detector

# Ekspor model CRAFT ke ONNX
dummy_input = torch.randn(1, 3, 640, 640)  # Sesuaikan dimensi input
onnx_craft_path = "craft_model.onnx"

torch.onnx.export(
    craft_model,
    dummy_input,
    onnx_craft_path,
    opset_version=11,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

print(f"Model CRAFT berhasil diekspor ke {onnx_craft_path}")


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


OnnxExporterError: Module onnx is not installed!

2. Akses dan Ekspor Recognition Model:

In [3]:
# Akses recognition model dari EasyOCR
recognition_model = reader.recognizer

# Ekspor recognition model ke ONNX
dummy_input = torch.randint(0, 256, (1, 512), dtype=torch.long)  # Contoh input untuk recognition
onnx_recognition_path = "recognition_model.onnx"

torch.onnx.export(
    recognition_model,
    dummy_input,
    onnx_recognition_path,
    opset_version=11,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)

print(f"Recognition model berhasil diekspor ke {onnx_recognition_path}")


TypeError: Model.forward() missing 1 required positional argument: 'text'

ONNX KE TENSORFLOW

In [ ]:
from onnx_tf.backend import prepare
import onnx

# Muat model ONNX
onnx_model = onnx.load("easyocr_model.onnx")

# Konversi ke TensorFlow
tf_rep = prepare(onnx_model)
tf_rep.export_graph("easyocr_model_tf")
print("Model berhasil dikonversi ke TensorFlow!")


Simpan Model TensorFlow sebagai .h5

In [ ]:
import tensorflow as tf

# Muat model TensorFlow
model = tf.keras.models.load_model("easyocr_model_tf")

# Simpan sebagai .h5
model.save("easyocr_model.h5")
print("Model berhasil disimpan sebagai .h5")

 Model .h5 untuk OCR

In [ ]:
from tensorflow.keras.models import load_model

# Fungsi untuk melakukan OCR menggunakan model .h5
def ocr_with_h5_model(image, model):
    processed_image = cv2.resize(image, (640, 640))  # Preprocessing gambar
    processed_image = processed_image / 255.0
    processed_image = processed_image.reshape(1, 640, 640, 3)

    # Prediksi menggunakan model
    predictions = model.predict(processed_image)
    return decode_predictions(predictions)  # Implementasikan fungsi decode jika perlu
